# Tool 1: Orderbook Time Series Viewer

## Purpose
Visualizes the full orderbook (bid/ask price levels with volumes) over time,
with trade markers overlaid. This is the single most important visualization
tool — it replicates the core dashboard used by the Frankfurt Hedgehogs
(2nd place, Prosperity 3) to identify every major trading pattern.

## What You'll See
- **Blue dots** = bid (buy) price levels, sized by volume
- **Red dots** = ask (sell) price levels, sized by volume
- **Green triangles** = executed trades
- **Orange dashed line** = Wall Mid (fair price estimate)
- **Gray dashed line** = Raw mid-price (for comparison)

## How to Use
1. Set the filter parameters in the **Configuration** cell below
2. Run all cells (Shift+Enter through the notebook)
3. The main plot shows the orderbook over time for your chosen product/day
4. Zoom into specific time windows by adjusting `TIME_START` / `TIME_END`
5. Toggle indicators on/off with the `SHOW_*` flags

## Key Patterns to Look For
- **Walls**: Thick clusters of blue/red dots at consistent price levels = market maker walls
- **Trades at extremes**: Triangles clustering at daily min/max prices = possible informed trader ("Olivia")
- **Spread tightening/widening**: Gap between blue and red clusters changing over time
- **Price drift vs stability**: Does the price level move or stay fixed?

## Dependencies
- `data_loader.py` (in same directory)
- `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION — Change these parameters to explore the data
# ============================================================

# Which product to display
# Options: "ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"
PRODUCT = "ASH_COATED_OSMIUM"

# Which day to display
# Options: -2, -1, 0
DAY = -1

# Time range filter (timestamp units, 0 to ~999900)
# Set TIME_END = None to show the full day
TIME_START = 0
TIME_END = None  # e.g., 100000 to zoom into the first 10% of the day

# --- Indicator toggles ---
SHOW_WALL_MID = True    # Orange dashed line: fair price from deepest liquidity
SHOW_RAW_MID = False    # Gray dashed line: simple (best_bid + best_ask) / 2
SHOW_TRADES = True      # Green triangles: executed trades

# --- Volume scaling ---
# Dot size is proportional to volume. Adjust this multiplier if dots are
# too large or too small on your screen.
VOLUME_SCALE = 3

# --- Figure size ---
FIG_WIDTH = 20
FIG_HEIGHT = 8

In [ ]:
# ============================================================
# SETUP — Imports and data loading (no need to modify)
# ============================================================
import sys
from pathlib import Path

# Ensure data_loader is importable
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent / "analysis"))
sys.path.insert(0, str(Path(__file__).parent if '__file__' in dir() else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from data_loader import (
    load_prices, load_trades, compute_wall_mid, compute_raw_mid,
    filter_time_range, PRODUCTS, AVAILABLE_DAYS,
)

print(f"Loading data: product={PRODUCT}, day={DAY}")
prices = load_prices(day=DAY, product=PRODUCT)
prices = compute_wall_mid(prices)
prices = compute_raw_mid(prices)
trades = load_trades(day=DAY, product=PRODUCT)

# Apply time range filter
if TIME_END is not None:
    prices = filter_time_range(prices, TIME_START, TIME_END)
    trades = filter_time_range(trades, TIME_START, TIME_END)
elif TIME_START > 0:
    prices = filter_time_range(prices, TIME_START, prices["timestamp"].max())
    trades = filter_time_range(trades, TIME_START, trades["timestamp"].max())

print(f"Loaded {len(prices)} orderbook snapshots, {len(trades)} trades")
print(f"Timestamp range: {prices['timestamp'].min()} — {prices['timestamp'].max()}")

In [ ]:
# ============================================================
# MAIN PLOT — Orderbook time series with trades and indicators
# ============================================================

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

ts = prices["timestamp"].values

# --- Plot bid levels (blue) ---
for level in [1, 2, 3]:
    p_col = f"bid_price_{level}"
    v_col = f"bid_volume_{level}"
    mask = prices[p_col].notna()
    if mask.any():
        ax.scatter(
            ts[mask], prices.loc[mask, p_col],
            s=prices.loc[mask, v_col] * VOLUME_SCALE,
            c="tab:blue", alpha=0.4, edgecolors="none",
            label=f"Bid L{level}" if level == 1 else None,
        )

# --- Plot ask levels (red) ---
for level in [1, 2, 3]:
    p_col = f"ask_price_{level}"
    v_col = f"ask_volume_{level}"
    mask = prices[p_col].notna()
    if mask.any():
        ax.scatter(
            ts[mask], prices.loc[mask, p_col],
            s=prices.loc[mask, v_col] * VOLUME_SCALE,
            c="tab:red", alpha=0.4, edgecolors="none",
            label=f"Ask L{level}" if level == 1 else None,
        )

# --- Plot trades (green triangles) ---
if SHOW_TRADES and len(trades) > 0:
    ax.scatter(
        trades["timestamp"], trades["price"],
        s=trades["quantity"] * VOLUME_SCALE * 3,
        c="tab:green", marker="^", alpha=0.7, edgecolors="black", linewidths=0.5,
        label="Trades", zorder=5,
    )

# --- Plot Wall Mid indicator ---
if SHOW_WALL_MID:
    ax.plot(
        ts, prices["wall_mid"],
        color="orange", linestyle="--", linewidth=1.2, alpha=0.8,
        label="Wall Mid (fair price)",
    )

# --- Plot Raw Mid indicator ---
if SHOW_RAW_MID:
    ax.plot(
        ts, prices["raw_mid"],
        color="gray", linestyle="--", linewidth=1, alpha=0.6,
        label="Raw Mid",
    )

ax.set_xlabel("Timestamp", fontsize=12)
ax.set_ylabel("Price", fontsize=12)
ax.set_title(f"Orderbook Time Series — {PRODUCT} (Day {DAY})", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSummary statistics for {PRODUCT} on Day {DAY}:")
print(f"  Wall Mid range: {prices['wall_mid'].min():.1f} — {prices['wall_mid'].max():.1f}")
print(f"  Wall Mid std:   {prices['wall_mid'].std():.2f}")
print(f"  Raw Mid range:  {prices['raw_mid'].min():.1f} — {prices['raw_mid'].max():.1f}")
print(f"  Avg spread:     {(prices['ask_price_1'] - prices['bid_price_1']).mean():.2f}")
print(f"  Total trades:   {len(trades)}")

## Quick Compare: All Products on One Day

Run the cell below to see both products side by side for the selected day.
This helps you quickly classify which product is fixed-price vs. moving.

In [ ]:
# ============================================================
# SIDE-BY-SIDE — Both products on the selected day
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

for idx, prod in enumerate(PRODUCTS):
    ax = axes[idx]
    p = compute_wall_mid(load_prices(day=DAY, product=prod))
    t = load_trades(day=DAY, product=prod)

    ts = p["timestamp"].values
    for level in [1, 2, 3]:
        mask = p[f"bid_price_{level}"].notna()
        if mask.any():
            ax.scatter(ts[mask], p.loc[mask, f"bid_price_{level}"],
                       s=p.loc[mask, f"bid_volume_{level}"] * VOLUME_SCALE,
                       c="tab:blue", alpha=0.3, edgecolors="none")
        mask = p[f"ask_price_{level}"].notna()
        if mask.any():
            ax.scatter(ts[mask], p.loc[mask, f"ask_price_{level}"],
                       s=p.loc[mask, f"ask_volume_{level}"] * VOLUME_SCALE,
                       c="tab:red", alpha=0.3, edgecolors="none")

    if len(t) > 0:
        ax.scatter(t["timestamp"], t["price"],
                   s=t["quantity"] * VOLUME_SCALE * 3,
                   c="tab:green", marker="^", alpha=0.6, edgecolors="black", linewidths=0.3)

    ax.plot(ts, p["wall_mid"], color="orange", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(f"{prod} (Day {DAY})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()